# SemanticRAG vs. VKG — Query-Agent Evaluation Comparison

This notebook compares the two semantic-layer **query paths** over the *same* normalized S3
(Iceberg) source data:

1. **Run A — SemanticRAG**: take a SemanticRAG layer built by the **metadata** enrichment agent
   (`MetadataRuntimeArn`), then evaluate the **metadata query agent**
   (`MetadataQueryRuntimeArn`) over the full groundtruth dataset.
2. **Run B — VKG**: take a VKG layer built by the **ontology** agent (`OntologyRuntimeArn`,
   N-Quads → Neptune), then evaluate the **ontology query agent** (`QueryRuntimeArn`) over the
   same groundtruth dataset.

For each run it captures, per question: **accuracy** (groundtruth LLM-as-Judge), **latency**,
and **cost** (input/output token usage). Finally it prints a side-by-side comparison.

> **Layer reuse (default):** what this notebook actually *measures* is the **query** agents —
> building a layer is only a prerequisite. So by **default it REUSES the most recent completed
> SemanticRAG / VKG layer** (built by notebooks 1 & 5, or a prior run of this notebook) rather
> than rebuilding. This makes a run fast (just the per-question query invocations) and lets the
> groundtruth questions resolve against the full 40-table layers. Set `REUSE_EXISTING_LAYER=0`
> to build fresh layers here instead (the old behaviour), or pin a specific layer with
> `SEMANTIC_RAG_LAYER_ID` / `VKG_LAYER_ID`.

> **Cost & time warning (build mode only):** with `REUSE_EXISTING_LAYER=0`, each run builds a
> *real* semantic layer (enrichment/ontology over N tables, plus KB ingestion for SemanticRAG)
> before querying. Use `MAX_TABLES` and `MAX_ROWS` to keep a first build-mode pass small. The
> ontology query agent is especially slow (~1–3 min/question) regardless of mode.


In [1]:
# Prereq (uncomment if not already installed):
# !pip install bedrock-agentcore-starter-toolkit==0.3.9

## 1. Setup & Credentials

In [2]:
import os
import json
import uuid
import time
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

# Long read timeout: both query agents are synchronous; the ontology one can take minutes.
invoke_config = Config(read_timeout=900, connect_timeout=60,
                       retries={'max_attempts': 0, 'mode': 'standard'}, signature_version='v4')
config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
account_id = identity['Account']
print(f"✓ AWS Credentials Verified — account ...{account_id[-4:]}, region {region}, "
      f"role {identity['Arn'].split('/')[-1]}")


import re as _re

def _redact_account_ids(obj):
    """Recursively replace AWS account IDs embedded in ARN strings with XXXXXXXXXXXX."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return _re.sub(r'(arn:[^:]*:[^:]*:[^:]*:)\d{12}:', r'\1XXXXXXXXXXXX:', obj)
    return obj

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


✓ AWS Credentials Verified — account ...4087, region us-east-1, role huthmac-Isengard


In [3]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# The boto3 bedrock-agentcore client (SigV4) no longer works; use a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    auth_out = stack_outputs('auth')
    token_endpoint = auth_out['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_out['M2mClientId']
    lam = session.client('lambda')
    cfg = lam.get_function_configuration(FunctionName=f'{PROJECT_NAME}-mcp-tools')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


## 2. Resolve Stacks, Runtimes, KB & Source Coordinates

Both runs use the same normalized S3 source. We resolve all four agent runtimes plus the
SemanticRAG KB (for ingestion waits).

In [4]:
def stack_outputs(stack_suffix: str) -> dict:
    """Return {OutputKey: OutputValue} for '{PROJECT_NAME}-{suffix}', or {} if absent."""
    cfn = session.client('cloudformation', region_name=region)
    for name in (f'{PROJECT_NAME}-dev-{stack_suffix}', f'{PROJECT_NAME}-{stack_suffix}'):
        try:
            return {o['OutputKey']: o['OutputValue']
                    for o in cfn.describe_stacks(StackName=name)['Stacks'][0].get('Outputs', [])}
        except Exception:
            continue
    return {}

ac = stack_outputs('agentcore')
kb = stack_outputs('bedrock-kb')
datalake = stack_outputs('data-lake')

# SemanticRAG path
METADATA_RUNTIME_ARN = ac['MetadataRuntimeArn']
METADATA_QUERY_RUNTIME_ARN = ac['MetadataQueryRuntimeArn']
# VKG path
ONTOLOGY_RUNTIME_ARN = ac['OntologyRuntimeArn']
ONTOLOGY_QUERY_RUNTIME_ARN = ac['QueryRuntimeArn']

SEMANTIC_RAG_KB_ID = kb['SemanticRagKbId']
SEMANTIC_RAG_DATA_SOURCE_ID = kb['SemanticRagDataSourceId']

S3T_DATABASE = os.environ.get('S3T_DATABASE') or datalake.get('Namespace', 'normalized')
S3T_CATALOG = os.environ.get('S3T_CATALOG')
if not S3T_CATALOG:
    prefix = datalake.get('S3TablesFederatedCatalogName', 's3tablescatalog')
    S3T_CATALOG = f"{prefix}/{PROJECT_NAME}-dev-analytics-tables"

METADATA_TABLE = os.environ.get('ONTOLOGY_METADATA_TABLE') or stack_outputs('dynamodb').get('MetadataTableName', f'{PROJECT_NAME}-dev-metadata')

print("✓ Resolved coordinates:")
print(f"  SemanticRAG build/query = {METADATA_RUNTIME_ARN.rsplit('/',1)[-1]} / {METADATA_QUERY_RUNTIME_ARN.rsplit('/',1)[-1]}")
print(f"  VKG build/query         = {ONTOLOGY_RUNTIME_ARN.rsplit('/',1)[-1]} / {ONTOLOGY_QUERY_RUNTIME_ARN.rsplit('/',1)[-1]}")
print(f"  SemanticRAG KB / DS     = {SEMANTIC_RAG_KB_ID} / {SEMANTIC_RAG_DATA_SOURCE_ID}")
print(f"  Normalized S3           = catalog '{S3T_CATALOG}', namespace '{S3T_DATABASE}'")
print(f"  Metadata DynamoDB table = {METADATA_TABLE}")

✓ Resolved coordinates:
  SemanticRAG build/query = semantic_layer_dev_metadata-SpwY3TDOAV / semantic_layer_dev_metadata_query-6aPZJf6eWO
  VKG build/query         = semantic_layer_dev_ontology-iddx7A6C74 / semantic_layer_dev_ontology_query-HKGVpkE6XK
  SemanticRAG KB / DS     = RAEDBATFHV / QK85B0M9YX
  Normalized S3           = catalog 's3tablescatalog/semantic-layer-dev-analytics-tables', namespace 'normalized'
  Metadata DynamoDB table = semantic-layer-dev-metadata


In [5]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# The boto3 bedrock-agentcore client (SigV4) no longer works; use a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    auth_out = stack_outputs('auth')
    token_endpoint = auth_out['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_out['M2mClientId']
    lam = session.client('lambda')
    cfg = lam.get_function_configuration(FunctionName=f'{PROJECT_NAME}-mcp-tools')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


## 3. Load the Groundtruth Dataset

`MAX_ROWS` slices the dataset for a quick pass (set 0 for all). Because the VKG query agent is
slow, keep this small for a first run.

In [6]:
with open('../data/eval/groundtruth_dataset.json', 'r') as f:
    groundtruth_all = json.load(f)

# Advisory rows (Expected_Tier == 'advisory') are answered by the intent-router
# advisory path and carry NO SQL — exempt them from the SQL-column requirement
# (mirrors nb2/nb6). BOTH query agents now answer advisory questions, so advisory
# rows are KEPT in the comparison (unlike nb9, which compares the SQL path only).
BASE_COLS = {'Natural_Language_Question', 'Expected_Answer'}
SQL_COLS = {'Expected_SQL_Query', 'Expected_SQL_Result'}
for i, row in enumerate(groundtruth_all):
    required = BASE_COLS if row.get('Expected_Tier') == 'advisory' else (BASE_COLS | SQL_COLS)
    missing = required - set(row.keys())
    if missing:
        raise ValueError(f"Row {i} missing columns: {missing}")

MAX_ROWS = int(os.environ.get('MAX_ROWS', '2'))
groundtruth = groundtruth_all[:MAX_ROWS] if MAX_ROWS > 0 else list(groundtruth_all)
print(f"✓ Loaded {len(groundtruth_all)} row(s); evaluating {len(groundtruth)} "
      f"({'slice' if MAX_ROWS>0 else 'full dataset'})")
display(pd.DataFrame(groundtruth)[['Natural_Language_Question']])

✓ Loaded 16 row(s); evaluating 16 (full dataset)


,Natural_Language_Question
0,Show me policies where the insured party is also the policyholder.
1,"For each rider, who are the insured participants and what are their roles?"
2,List the top 5 most common party types and their human-readable descriptions.
3,"What is the total market value of all active holdings, grouped by party?"
4,"For policies that have a payout schedule, show the policyholder's name, the payout frequency, and the projected next-payout amount."
5,How many parties are there?
6,List the top 10 coverage products by name.
7,"Show the top 10 parties by total holding market value, including the investment product names they hold."
8,What was the total financial-activity amount per month in 2024?
9,How many are there?


## 4. Discover Normalized S3 Source Tables

Both layers are built over the same tables. `MAX_TABLES` caps how many (0 = all).

In [7]:
glue = session.client('glue', region_name=region)
MAX_TABLES = int(os.environ.get('MAX_TABLES', '3'))

s3_catalog_id = f"{account_id}:{S3T_CATALOG}"
s3_tables = []
for page in glue.get_paginator('get_tables').paginate(CatalogId=s3_catalog_id, DatabaseName=S3T_DATABASE):
    s3_tables.extend(t['Name'] for t in page.get('TableList', []))
s3_tables = sorted(s3_tables)
if MAX_TABLES > 0:
    s3_tables = s3_tables[:MAX_TABLES]
print(f"✓ Normalized S3: {len(s3_tables)} table(s): {s3_tables}")


def s3_source(table_name: str) -> dict:
    """Data-source dict for a normalized S3 Tables (Iceberg) table."""
    return {
        'databaseName': S3T_DATABASE,
        'tableName':    table_name,
        'catalogId':    S3T_CATALOG,
        'dataSource':   'AwsDataCatalog',
        'tableId':      f'{S3T_CATALOG}::{S3T_DATABASE}.{table_name}',
    }

SOURCES = [s3_source(t) for t in s3_tables]

✓ Normalized S3: 40 table(s): ['address', 'admin_codes', 'annuity_detail', 'arrangement_destination', 'arrangement_source', 'carrier_appointment', 'coverage', 'coverage_product', 'distribution_level', 'email_address', 'financial_activity', 'financial_statement', 'govt_id_info', 'holding', 'holding_activity', 'holding_arrangement', 'holding_dbg', 'holding_loan', 'holding_payout', 'holding_projection', 'holding_restriction', 'holding_subaccount', 'invest_product', 'life_detail', 'life_participant', 'loan_activity', 'party', 'party_banking', 'party_license', 'phone', 'policy_loan_summary', 'policy_product', 'producer_agreement', 'reinsurance_info', 'relation', 'rider', 'rider_participant', 'subaccount_activity', 'substandard_rating', 'type_codes']


## 5. Reusable Pipeline Helpers

- `build_layer(...)` — seed a config (type SemanticRAG or VKG), invoke the matching build
  agent, poll DynamoDB to `completed`.
- `wait_for_kb_ingestion(...)` — for the SemanticRAG path, wait for the auto-triggered KB
  ingestion job (the VKG path persists to Neptune synchronously, no KB ingestion).
- `judge_row(...)` — the groundtruth text-to-SQL judge (identical rubric across notebooks).
- `run_query_eval(...)` — invoke a query agent once per row, judge each, collect accuracy /
  latency / token metrics. Works for both query agents (both return answer/sql_query/results,
  and the ontology agent additionally returns metadata.usage).

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Thin comparator — NO build, NO eval, NO runtime invocation here.
#
# Both arms of this comparison are ALREADY evaluated k times by their own notebooks:
#   - SemanticRAG arm  → notebook 2 (metadata query agent)  → metadata_query_kmean_eval_*.json
#   - VKG arm          → notebook 6 (ontology query agent)  → ontology_query_kmean_eval_*.json
# Re-running either agent here would just repeat that work, so notebook 10 simply LOADS the
# two most-recent k-run mean files and compares them. Run notebooks 2 and 6 first.
# ════════════════════════════════════════════════════════════════════════════
import glob as _glob


def load_latest_kmean(prefix: str, *, label: str) -> dict:
    """Load the most recent k-run mean file results/{prefix}_*.json, tagged with `label`.

    Parameters:
        prefix: filename prefix, e.g. 'metadata_query_kmean_eval' (nb2) or
            'ontology_query_kmean_eval' (nb6).
        label: human arm tag for the comparison row ('semantic-rag' / 'vkg').
    Returns:
        dict with label, mean_scores, agent_perf_mean (token key normalised),
        per_scenario_goal_mean, eval_id, eval_k, source_file.
    Raises:
        FileNotFoundError if no matching file exists — fail loudly so a missing upstream run
        (notebook 2 or 6 not run yet) is obvious rather than silently producing a half table.
    """
    matches = sorted(_glob.glob(f"../data/eval/results/{prefix}_*.json"))
    if not matches:
        raise FileNotFoundError(
            f"No {prefix}_*.json found in ../data/eval/results/. Run the upstream notebook "
            f"first (notebook 2 for the SemanticRAG arm, notebook 6 for the VKG arm).")
    path = matches[-1]
    payload = json.load(open(path))
    # Normalise the token key: nb2's kmean carries 'total_tokens'; nb6/nb9 carry
    # 'agent_total_tokens'. Expose 'agent_total_tokens' uniformly for the compare cell.
    apm = dict(payload.get('agent_perf_mean', {}) or {})
    if 'agent_total_tokens' not in apm and 'total_tokens' in apm:
        apm['agent_total_tokens'] = apm['total_tokens']
    print(f"  ♻ [{label}] loaded {os.path.basename(path)} "
          f"(eval_k={payload.get('eval_k')}, scores={payload.get('mean_scores')})")
    return {'label': label,
            'mean_scores': payload.get('mean_scores', {}) or {},
            'std_scores': payload.get('std_scores', {}) or {},
            'agent_perf_mean': apm,
            'per_scenario_goal_mean': payload.get('per_scenario_goal_mean', {}) or {},
            'eval_id': payload.get('eval_id'), 'eval_k': payload.get('eval_k'),
            'source_file': os.path.basename(path)}


print("✓ Comparator ready — load_latest_kmean (no build / eval / runtime calls in nb10)")


In [ ]:
# ── No layer build / reuse switch in notebook 10 ────────────────────────────
# Notebook 10 is now a pure comparator: it reads notebook 2's and notebook 6's k-run mean
# files. The layers those notebooks evaluated (SemanticRAG via nb2, VKG via nb6) are recorded
# in the loaded files' eval_id, so there is nothing to build or reuse here.
print("ℹ Notebook 10 reads notebook 2 + notebook 6 k-run means; no layer build happens here.")


## 6. Run A — SemanticRAG Path

Build a SemanticRAG layer (metadata agent) → wait for KB ingestion → eval the metadata query
agent.

In [ ]:
print("=== RUN A: SemanticRAG (reused from notebook 2's k-run mean) ===")
# The SemanticRAG arm = the metadata query agent, which notebook 2 already evaluates EVAL_K
# times over the normalized-S3 layer. Load its k-run mean file (no re-run here).
kmean_a = load_latest_kmean('metadata_query_kmean_eval', label='semantic-rag')
print(f"\n  Run A k-run MEAN scores (eval_k={kmean_a.get('eval_k')}): {kmean_a['mean_scores']}")


## 7. Run B — VKG Path

Build a VKG layer (ontology agent → Neptune) → eval the ontology query agent. No KB ingestion
wait — the ontology agent persists triples to Neptune as part of its build.

In [ ]:
print("=== RUN B: VKG (reused from notebook 6's k-run mean) ===")
# The VKG arm = the ontology query agent, which notebook 6 already evaluates EVAL_K times over
# the best-coverage VKG layer. Load its k-run mean file (no re-run here).
kmean_b = load_latest_kmean('ontology_query_kmean_eval', label='vkg')
print(f"\n  Run B k-run MEAN scores (eval_k={kmean_b.get('eval_k')}): {kmean_b['mean_scores']}")


## 8. Compare the Two Query Paths

Aggregate accuracy, latency, and cost (tokens) per path.

In [ ]:
_EVALS = ['Builtin.GoalSuccessRate',
          'FinalAnswerFaithfulness', 'SqlGrounded', 'ToolCallOrdering']


def summarize_kmean(kmean: dict) -> dict:
    """Flatten one path's k-run MEAN result into a comparison row.

    `kmean` is the dict from load_latest_kmean — per-evaluator mean scores in `mean_scores`,
    plus the agent's own mean latency/tokens in `agent_perf_mean`.
    """
    scores = kmean.get('mean_scores', {}) or {}
    perf = kmean.get('agent_perf_mean', {}) or {}
    row = {'path': kmean.get('label'), 'eval_k': kmean.get('eval_k')}
    row.update({ev: scores.get(ev) for ev in _EVALS})
    row['avg_latency_s'] = perf.get('avg_wall_clock_s')
    row['agent_total_tokens'] = perf.get('agent_total_tokens')
    return row


comparison = pd.DataFrame([summarize_kmean(kmean_a), summarize_kmean(kmean_b)])
print("=== COMPARISON (k-run MEAN): SemanticRAG (nb2) vs VKG (nb6), same groundtruth dataset ===")
display(comparison)

os.makedirs('../data/eval/results', exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out = f'../data/eval/results/semantic_rag_vs_vkg_kmean_{ts}.json'
with open(out, 'w') as f:
    json.dump(
        _redact_account_ids({
            'eval_k': {'semantic-rag': kmean_a.get('eval_k'), 'vkg': kmean_b.get('eval_k')},
            'sources': {'semantic-rag': kmean_a.get('source_file'),
                        'vkg': kmean_b.get('source_file')},
            'eval_ids': {'semantic-rag': kmean_a.get('eval_id'), 'vkg': kmean_b.get('eval_id')},
            'kmean_a': kmean_a, 'kmean_b': kmean_b,
            'comparison': comparison.to_dict(orient='records'),
        }),
        f, indent=2, default=str,
    )
print(f"✓ Wrote {out}")

# Headline — GoalSuccessRate (k-run mean) is the primary accuracy signal; SqlGrounded /
# ToolCallOrdering on the VKG arm sit in the documented harvest-noise band (judge VKG by
# GoalSuccess + answer inspection).
a_s, b_s = kmean_a.get('mean_scores', {}) or {}, kmean_b.get('mean_scores', {}) or {}
a_gsr, b_gsr = a_s.get('Builtin.GoalSuccessRate'), b_s.get('Builtin.GoalSuccessRate')
if a_gsr is not None and b_gsr is not None:
    winner = 'vkg' if b_gsr > a_gsr else ('semantic-rag' if a_gsr > b_gsr else 'tie')
    a_perf, b_perf = kmean_a.get('agent_perf_mean', {}), kmean_b.get('agent_perf_mean', {})
    headline = (
        "### Headline (k-run mean)\n"
        f"- **Runs averaged:** semantic-rag {kmean_a.get('eval_k')} (nb2) · vkg {kmean_b.get('eval_k')} (nb6)\n"
        f"- **GoalSuccessRate:** semantic-rag {a_gsr:.1%} vs vkg {b_gsr:.1%} → **{winner}**\n"
        f"- **FinalAnswerFaithfulness:** semantic-rag {a_s.get('FinalAnswerFaithfulness')} vs vkg {b_s.get('FinalAnswerFaithfulness')}\n"
        f"- **SqlGrounded:** semantic-rag {a_s.get('SqlGrounded')} vs vkg {b_s.get('SqlGrounded')}\n"
        f"- **ToolCallOrdering:** semantic-rag {a_s.get('ToolCallOrdering')} vs vkg {b_s.get('ToolCallOrdering')}\n"
        f"- **Avg latency:** semantic-rag {a_perf.get('avg_wall_clock_s')}s vs vkg {b_perf.get('avg_wall_clock_s')}s\n"
        f"- **Cost (agent tokens):** semantic-rag {a_perf.get('agent_total_tokens')} vs vkg {b_perf.get('agent_total_tokens')}\n"
        f"\n_VKG's SqlGrounded / FinalAnswerFaithfulness sit in the documented harvest-noise "
        f"band; judge VKG primarily by GoalSuccessRate + answer inspection (see notebook 6)._"
    )
    display(Markdown(headline))
else:
    print("⚠ One or both arms produced no aggregate scores — comparison is incomplete. "
          "Run notebooks 2 and 6 first.")


## Summary

Compares the **SemanticRAG** and **VKG** query paths over the same normalized S3 source data,
on accuracy / latency / cost.

### Knobs
- `REUSE_EXISTING_LAYER` (default `1`) — reuse the most recent completed layer of each type
  instead of rebuilding. Set `0` to build fresh layers in this notebook.
- `SEMANTIC_RAG_LAYER_ID` / `VKG_LAYER_ID` — pin a specific config `id` to reuse (must be of
  the matching type and `completed`). Overrides the "most recent" auto-pick.
- `MAX_TABLES` (default 0) — tables per layer **when building** (`REUSE_EXISTING_LAYER=0`);
  ignored when reusing. 0 = all.
- `MAX_ROWS` (default 0) — groundtruth questions per run; 0 = all.

### Notes / caveats
- **Reuse is the default** because the notebook measures the *query* agents, not the build
  agents; reusing also lets queries resolve against the full 40-table layers from notebooks
  1 & 5. Reuse skips the KB ingestion wait (the reused layer was ingested at build time).
- The VKG query agent is much slower (graph retrieval + SQL synthesis), so latency/cost are
  expected to differ markedly — that's part of what this notebook surfaces.